# Groq policy responses for PASS, FAIL, and REVIEW cases

This notebook calls Groq once for each synthetic case. It reads the policy at runtime from `loan_policy.md`; it does not duplicate numeric policy rules in the prompt. Knowlytix `ClaimVerifier` computes the actual PASS / FAIL / REVIEW status before Groq is asked to explain it.

**These are live API calls and may use Groq quota.** The response is informational only. A human decision is always required. The geometric `ConstrainedLLM` claim gate runs only if the GMS tension cuts have valid labeled-pair calibration; this project's current calibration is degenerate, so the notebook will report that gate unavailable and keep all drafts blocked for human review instead of substituting made-up thresholds.


In [29]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import loanguard as loan

load_dotenv(PROJECT / ".env", override=False)
if not os.getenv("GROQ_API_KEY"):
    raise RuntimeError("Set GROQ_API_KEY in the project .env or process environment.")
print("Groq key detected; value is hidden.")


Groq key detected; value is hidden.


## 1. Check the live model catalog


In [30]:
catalog = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    timeout=30,
)
catalog.raise_for_status()
groq_model_ids = sorted(item["id"] for item in catalog.json()["data"])
print("Models available to this key:")
for model_id in groq_model_ids:
    print(" -", model_id)


Models available to this key:
 - allam-2-7b
 - canopylabs/orpheus-arabic-saudi
 - canopylabs/orpheus-v1-english
 - meta-llama/llama-prompt-guard-2-22m
 - meta-llama/llama-prompt-guard-2-86m
 - openai/gpt-oss-120b
 - openai/gpt-oss-20b
 - openai/gpt-oss-safeguard-20b
 - qwen/qwen3.8-27b
 - whisper-large-v3
 - whisper-large-v3-turbo


In [31]:
from knowlytix.knowledge.llm_backend import LiteLLMBackend

requested_model = (os.getenv("KNOWLYTIX_GROQ_MODEL") or "").strip().removeprefix("groq/")
if requested_model:
    raw_model_id = requested_model
else:
    preferred_models = [
        "openai/gpt-oss-20b", "llama-3.3-70b-versatile",
        "llama-3.1-8b-instant", "qwen/qwen3-32b",
    ]
    raw_model_id = next((model for model in preferred_models if model in groq_model_ids), None)
    if raw_model_id is None:
        raise RuntimeError(
            "No preferred chat model is available. Set KNOWLYTIX_GROQ_MODEL "
            "to an exact model ID from the catalog and rerun this cell."
        )
    print("No model override set; selected available model:", raw_model_id)
if raw_model_id not in groq_model_ids:
    raise ValueError(f"{raw_model_id!r} is not available in the live catalog.")
groq_llm = LiteLLMBackend(f"groq/{raw_model_id}")
print("Configured Knowlytix backend:", groq_llm.model_name)


No model override set; selected available model: openai/gpt-oss-20b
Configured Knowlytix backend: groq/openai/gpt-oss-20b


## 2. Load policy knowledge and calibrate the verifier

The same calibration from `build_verifier()` is used for typed policy claims and to configure the optional geometric gate. If tension calibration is unusable, the geometric gate is skipped explicitly; a theoretical or hardcoded threshold is never substituted.


In [ ]:
import importlib
loan = importlib.reload(loan)  # Refresh helper code if this kernel imported an older version.

from knowlytix.harness._verifier import LogicVerifier
from knowlytix.harness.governance.governed_llm import ConstrainedLLM
from knowlytix.knowledge.config import VerifyConfig

store = loan.build_or_load_store(rebuild=False, epochs=100)
claim_verifier = loan.build_verifier(store)
tension_calibration = getattr(claim_verifier, "calibrated_thresholds", None)
if tension_calibration is None:
    raise RuntimeError(
        "Calibration metadata is missing. Restart the kernel and run notebook cells from the top; "
        "the current loanguard.build_verifier() must attach calibrated_thresholds."
    )
print({
    "tension_status": tension_calibration.tension_status,
    "tau_ent": tension_calibration.tau_ent,
    "tau_contra": tension_calibration.tau_contra,
    "tau_plausibility": tension_calibration.tau_plausibility,
})

constrained = None
if tension_calibration.tension_ok and tension_calibration.tau_plausibility is not None:
    logic_verifier = LogicVerifier(
        store,
        VerifyConfig(
            tau_ent=tension_calibration.tau_ent,
            tau_contra=tension_calibration.tau_contra,
            min_plausibility=tension_calibration.tau_plausibility,
        ),
    )
    constrained = ConstrainedLLM(
        store=store, verifier=logic_verifier, llm=groq_llm,
        allowed_tools=["lookup_product_terms"],
    )
    print("Geometric claim gate: configured from calibrated Knowlytix thresholds.")
else:
    print(
        "Geometric claim gate: unavailable because tension cuts are not calibrated. "
        "Drafts will remain blocked for human review."
    )




## 3. Run the three examples through the policy checks

The sample values are scenario inputs. The policy limits are read from the GMS ENM entries created from `loan_policy.md`. A failed pre-screen is referred to manual review; it is not a final loan decline.


In [ ]:
examples = [
    {"example": "PASS", "applicant_id": "DEMO-PASS", "product": "Personal Loan",
     "requested_amount": 12000, "credit_score": 712, "proposed_apr": 16.99},
    {"example": "FAIL", "applicant_id": "DEMO-FAIL", "product": "Personal Loan",
     "requested_amount": 28000, "credit_score": 735, "proposed_apr": 18.50},
    {"example": "REVIEW", "applicant_id": "DEMO-REVIEW", "product": "Personal Loan",
     "requested_amount": 15000, "credit_score": 700, "proposed_apr": None},
]
example_results = []
for example in examples:
    screened = loan.screen_application(store, claim_verifier, example)
    example_results.append({**example, **screened})

status_rows = []
for result in example_results:
    checks = result["checks"]
    status_rows.append({
        "example": result["example"],
        "screening_status": result["screening_status"],
        "screening_result": result["screening_result"],
        "amount_check": checks["requested_amount_within_policy_maximum"]["status"],
        "score_check": checks["credit_score_meets_policy_minimum"]["status"],
        "APR_check": checks["proposed_apr_within_policy_maximum"]["status"],
        "income_verification_required": result["income_verification_required"],
        "routing": result["routing"],
        "human_decision_required": result["human_decision_required"],
    })
pd.DataFrame(status_rows)


## 4. Ask Groq to explain each result using the Markdown policy

Each request contains the full policy text and the actual Knowlytix check evidence for that example. The prompt asks for one short paragraph to reduce truncation. `LiteLLMBackend.call()` returns text but not Groq `finish_reason`, so the punctuation indicator below is only a visual warning, not authoritative completion metadata.


In [ ]:
policy_text = (PROJECT / "loan_policy.md").read_text(encoding="utf-8-sig")
drafts = []
for result in example_results:
    checks = result["checks"]
    expected = {
        "example": result["example"],
        "screening_status": result["screening_status"],
        "screening_result": result["screening_result"],
        "amount_check": checks["requested_amount_within_policy_maximum"]["status"],
        "score_check": checks["credit_score_meets_policy_minimum"]["status"],
        "apr_check": checks["proposed_apr_within_policy_maximum"]["status"],
        "income_verification_required": result["income_verification_required"],
        "routing": result["routing"],
        "human_decision_required": result["human_decision_required"],
    }
    scenario = {
        "application": {
            "product": result["product"],
            "requested_amount": result["requested_amount"],
            "credit_score": result["credit_score"],
            "proposed_apr": result["proposed_apr"],
        },
        "knowlytix_verified_result": expected,
        "policy_source": policy_text,
    }
    try:
        response = groq_llm.call(
            system=(
                "Return one valid JSON object only, without Markdown fences. Copy every "
                "screening and check field exactly from knowlytix_verified_result; do "
                "not reinterpret, combine, or change any status. Add one short explanation "
                "sentence based only on the supplied policy and verified result. "
                "Make it case-specific by naming checks that passed, failed, or need review. "
                "A PASS means eligible for underwriting, never final approval; a FAIL is "
                "not a final decline. Do not call any role the sole authority or invent "
                "approval powers beyond the policy's Approval Authority field."
            ),
            user=(
                "Explain this case. Return exactly these fields: example, screening_status, "
                "screening_result, amount_check, score_check, apr_check, "
                "income_verification_required, routing, human_decision_required, explanation. "
                "Use the verified values exactly. Make the explanation vary with this "
                "applicant's actual check results; avoid a generic repeated sentence. "
                "Do not add facts absent from the policy or verified result.\n\n"
                + json.dumps(scenario, indent=2)
            ),
            max_tokens=1800,
        )
        parsed = json.loads(response.strip())
        mismatches = [key for key, value in expected.items() if parsed.get(key) != value]
        valid_json = True
    except Exception as exc:
        parsed = None
        mismatches = ["invalid_or_incomplete_json"]
        valid_json = False
        error = type(exc).__name__
    else:
        error = None
    drafts.append({
        "example": result["example"],
        "expected": expected,
        "parsed": parsed,
        "response": response if error is None else "",
        "valid_json": valid_json,
        "mismatched_fields": mismatches,
        "provider_error": error,
    })
    print(f"\n### {result['example']} ? policy status {result['screening_status']}")
    if error:
        print("Groq response could not be parsed:", error)
    else:
        print(json.dumps(parsed, indent=2))
        print("Status field mismatches:", mismatches or "none")

pd.DataFrame([
    {"example": row["example"], "valid_json": row["valid_json"],
     "mismatched_fields": row["mismatched_fields"], "provider_error": row["provider_error"]}
    for row in drafts
])


## 5. Apply the geometric claim gate only when calibrated

When Knowlytix has valid calibrated tension cuts, `ConstrainedLLM` extracts claims and applies its geometric gate, followed by an exact `ClaimVerifier` check against the policy graph. With the current `degenerate` calibration status, the notebook does not run that gate and does not release any Groq draft. All three drafts remain for human review. This avoids both the `ValueError` and the use of invented thresholds.


In [ ]:
# The applicant policy screen and the optional LLM-response release gate are
# separate decisions. An unavailable geometric gate must not erase the
# applicant-specific policy outcome shown below.
results_by_example = {result["example"]: result for result in example_results}
gate_rows = []
for item in drafts:
    result = results_by_example[item["example"]]
    status = result["screening_status"]
    if status == "PASS":
        policy_outcome = "POLICY_ELIGIBLE_FOR_UNDERWRITING"
    elif status == "FAIL":
        policy_outcome = "OUTSIDE_AUTO_PASS_LIMITS; ROUTE_TO_MANUAL_REVIEW"
    else:
        policy_outcome = "REVIEW_REQUIRED; ROUTE_TO_MANUAL_REVIEW"

    # The gate below controls release of generated explanatory text only.
    if constrained is None:
        geometric_gate = "UNAVAILABLE: tension calibration degenerate"
        response_release = "BLOCK_LLM_TEXT_AND_ESCALATE_TO_HUMAN"
    elif item["provider_error"] or not item["response"].strip():
        geometric_gate = "NOT_RUN: no response"
        response_release = "BLOCK_LLM_TEXT_AND_ESCALATE_TO_HUMAN"
    else:
        try:
            gated_claims = constrained.extract_claims(item["response"])
            print(f"Raw extractor output for {item['example']}: {gated_claims!r}")
            exact_results = []
            for gated in gated_claims:
                triple = gated.triple
                exact_claim = loan.Claim(
                    type=loan.ClaimType.FACT,
                    text=gated.text,
                    metadata={"head": triple.head, "relation": triple.relation, "tail": triple.tail},
                )
                exact = claim_verifier.verify_all([exact_claim]).claim_verdicts[0]
                exact_results.append(
                    bool(getattr(gated, "approved", False)
                         and exact.passed
                         and "exact graph match" in exact.evidence.casefold())
                )
            released = bool(exact_results) and all(exact_results)
            if not gated_claims:
                geometric_gate = "NO_CLAIMS_EXTRACTED"
                response_release = "BLOCK_LLM_TEXT_AND_ESCALATE_TO_HUMAN"
            else:
                geometric_gate = f"{sum(exact_results)}/{len(exact_results)} exact claims passed"
                response_release = "TEXT_GATE_PASSED" if released else "BLOCK_LLM_TEXT_AND_ESCALATE_TO_HUMAN"
        except ValueError as exc:
            geometric_gate = f"UNAVAILABLE: {type(exc).__name__}"
            response_release = "BLOCK_LLM_TEXT_AND_ESCALATE_TO_HUMAN"

    gate_rows.append({
        "example": item["example"],
        "screening_status": status,
        "applicant_policy_outcome": policy_outcome,
        "human_decision_required": result["human_decision_required"],
        "geometric_claim_gate": geometric_gate,
        "llm_response_release_status": response_release,
    })
pd.DataFrame(gate_rows)


## Interpretation

The typed threshold results are policy checks from Knowlytix; Groq only explains those results. PASS means eligible for the human underwriting stage, FAIL means rejected from the automatic pass path and referred to manual review, and REVIEW means a required value could not be checked. None is a final credit decision. With the current degenerate calibration, the geometric LLM claim gate stays fail-closed unless the store produces usable labeled-pair tension calibration.
